# 04 v2. 분할: 전술 / 기법 / 전술+기법 (A, B)

```
01_build_labels → preprocess → window → [이 노트북] → finetuning
```

**입력**: `data/interim/windows_{A,B}_size5_stride1.csv` (`window.ipynb` 결과)
- 같은 내용 윈도우는 `window.ipynb`가 이미 표시했다. 대표 윈도우(`is_first`)만 쓰고, 라벨은 묶음 합집합(`u_tactics`, `u_techniques`)을 쓴다 (README L6)
- 라벨 규칙(전술 각각, 기법 + 상위 기법, 기법 ID 정리)은 `01_build_labels`·`window.ipynb`에서 이미 적용됐다

**출력**: `data/v2/{A,B}_{tactic,technique,both}_k{1,3,5,8,16}/` 마다 (k마다 학습 라벨이 다르므로 폴더를 나눔)
- `labels.csv`: 이 k의 라벨 표 (역할, 이유, 칸별 개수)
- `splits/{이름}/pool.jsonl`, `val.jsonl`, `test.jsonl`, `val_gliclass.json`, `test_gliclass.json`, `kshot/train_k{k}_s{0,1,2}.json`
- `data/v2/roles_by_k.csv`: 방식·k마다 라벨 역할 전체

`finetuning.ipynb`는 `DATA_DIR=data/v2/B_tactic_k16`, `VARIANT='B_tactic_k16'`처럼 그대로 읽을 수 있다.

**⚠️ 문장 변환은 우선 건너뛴다.** 학습 파일의 `text`에는 임시로 `signature`(기호 문자열)를 넣는다. 문장 변환(03 방식)은 나중에 이 노트북의 2번에 넣는다.

## 0. 설정

| 설정 | 값 | 뜻 |
|---|---|---|
| 라벨 체계 | `labels/labels.csv` | 전술·기법 ID, 종류, 라벨 문장, 기본 역할. 묶은 라벨 없음 |
| `HELD_TACTICS` | `labels.csv`에서 `role = held_out`인 전술 (TA0040 Impact, TA0112 Defense Impairment) | 숨김 전술. 붙은 윈도우는 전부 test로 보낸다 (완전한 제로샷) |
| 기법 | 숨기지 않음 | 분할 뒤 개수 규칙으로만 숨김 |
| `AUTO_HIDE` | True | 분할 뒤 pool·val·test 중 한 곳이라도 0개인 라벨은 숨김으로 전환 |
| `KS`, `SEEDS` | 1, 3, 5, 8, 16 / 0, 1, 2 | k는 **윈도우 수** 기준. **pool에 k개가 안 되는 라벨은 그 k에서 숨김으로 전환** (학습 라벨은 항상 k개 이상) |

In [1]:
import io, json, os, shutil, subprocess
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 100)

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INTERIM = REPO / 'data' / 'interim'
OUT = REPO / 'data' / 'v2'

AUTO_HIDE = True
KS, SEEDS = [1, 3, 5, 8, 16], [0, 1, 2]
RATIO = {'pool': .50, 'val': .25, 'test': .25}
BIG = 20                       # 윈도우 20개 이상 세션은 세션 안에서 시간순으로 나눔
MODES = ['tactic', 'technique', 'both']

LABELS = pd.read_csv(REPO / 'labels' / 'labels.csv', keep_default_na=False).set_index('label_id')   # 라벨 체계 기준
assert not LABELS.index.str.contains(r'\+').any(), '묶은 라벨이 있음 (labels.csv)'
HELD_TACTICS = LABELS[(LABELS.kind == 'tactic') & (LABELS.role == 'held_out')].index.tolist()
print('숨김 전술:', HELD_TACTICS)
print('OUT =', OUT)

숨김 전술: ['TA0112', 'TA0040']
OUT = /home/openclaw/work2/ctgli/data/v2


## 1. 윈도우 불러오기

[하는 일] 대표 윈도우(`is_first`)만 쓰고 라벨은 묶음 합집합을 쓴다.
- `tactics`: 전술 각각 (`BENIGN` 가능, 정상·공격이 함께 있을 수 있음)
- `techniques`: 기법 + 상위 기법

[볼 것] A 9,287, B 7,733이면 `window.ipynb` 결과 그대로.

In [2]:
def split_labels(s):
    return sorted(set(str(s).split('|')) - {''})

data = {}
for k in 'AB':
    w = pd.read_csv(INTERIM / f'windows_{k}_size5_stride1.csv', low_memory=False, keep_default_na=False)
    w = w[w.is_first.astype(str) == 'True'].copy()
    w['tactics'] = w.u_tactics.map(split_labels)
    w['techniques'] = w.u_techniques.map(split_labels)
    w['window'] = w.window.astype(int)
    w['mixed'] = w.mixed.astype(str) == 'True'
    data[k] = w.reset_index(drop=True)
print({k: len(d) for k, d in data.items()})

{'A': 9287, 'B': 7733}


## 2. 문장 변환 (건너뜀)

**우선 건너뛴다.** `text`에 임시로 `signature`를 넣는다. 나중에 03의 `to_text`를 여기에 넣고, 다른 signature가 같은 문장이 되면 라벨 합집합으로 합친다 (L6).

In [3]:
for k, d in data.items():
    d['text'] = d['signature']          # TODO: 문장 변환
    assert d.text.is_unique
print('text = signature (임시)')

text = signature (임시)


## 3. 나누기 (규칙 1~3)

1. 숨김 전술(`HELD_TACTICS`)이 붙은 윈도우는 전부 test
2. 윈도우 20개 이상 세션은 세션 안에서 시간순 50/25/25
3. 20개 미만 세션은 통째로, 그 세션의 가장 흔한 전술 조합 기준 목표 비율에 가장 모자란 칸에

In [4]:
def split(d):
    d = d.copy(); d['split'] = None
    d['key'] = d.tactics.map('|'.join)
    held = d.tactics.map(lambda ls: any(l in HELD_TACTICS for l in ls))
    d.loc[held, 'split'] = 'test'                                       # 규칙 1
    rest = d[~held]
    size = rest.groupby('session').size()
    for s in size[size >= BIG].index:                                    # 규칙 2
        idx = rest[rest.session == s].sort_values('window').index
        c1, c2 = int(len(idx) * .50), int(len(idx) * .75)
        d.loc[idx[:c1], 'split'] = 'pool'; d.loc[idx[c1:c2], 'split'] = 'val'; d.loc[idx[c2:], 'split'] = 'test'
    small = size[size < BIG].sort_values(ascending=False).index          # 규칙 3
    main_key = rest.groupby('session').key.agg(lambda s: s.value_counts().index[0])
    total = rest.key.value_counts()
    for s in small:
        key = main_key[s]
        now = d[(d.key == key) & d.split.notna()].split.value_counts()
        need = {sp: RATIO[sp] * total[key] - now.get(sp, 0) for sp in RATIO}
        d.loc[(d.session == s) & d.split.isna(), 'split'] = max(need, key=need.get)
    assert d.split.notna().all()
    return d

parts = {k: split(data[k]) for k in 'AB'}
for k in 'AB':
    print(k, parts[k].split.value_counts().to_dict())

A {'pool': 4372, 'test': 2741, 'val': 2174}
B {'pool': 3607, 'test': 2319, 'val': 1807}


## 4. 겹치는 윈도우 버리기 (규칙 4)

같은 세션에서 윈도우 번호 차이가 5 미만이면 이벤트를 나눠 갖는다. 다른 칸으로 간 경우 아래 칸 쪽을 버린다 (test > val > pool).

In [5]:
RANK = {'pool': 0, 'val': 1, 'test': 2}

def purge(d):
    drop = []
    for s, g in d.groupby('session'):
        w = g.window.values; r = g.split.map(RANK).values
        for i, ix in enumerate(g.index):
            if ((np.abs(w - w[i]) < 5) & (r > r[i])).any():
                drop.append(ix)
    print(f'  겹쳐서 버림: {len(drop)}개', dict(d.loc[drop, 'split'].value_counts()))
    return d.drop(drop).reset_index(drop=True)

for k in 'AB':
    print(k); parts[k] = purge(parts[k])

A
  겹쳐서 버림: 98개 {'pool': np.int64(54), 'val': np.int64(44)}
B


  겹쳐서 버림: 145개 {'pool': np.int64(75), 'val': np.int64(70)}


## 5. 라벨 역할 정하기 (변환 규칙)

[하는 일] 라벨마다 pool / val / test 윈도우 수를 세고 역할을 정한다.
- `BENIGN`: 학습
- `labels.csv`에서 `held_out`인 라벨 (숨김 전술 2개): `held_out` (이유 `held_tactic`)
- 그 외: pool·val·test **한 곳이라도 0개면** `held_out` (이유 `no_pool` / `no_val` / `no_test`), 아니면 `train`

숨김(`held_out`) 라벨은 학습·검증 후보에서 빠지고 시험 후보에만 들어간다.
개수 규칙으로 숨긴 라벨은 그 윈도우가 pool·val에 남아 있을 수 있다 (정답에서만 빠짐). 그래서 숨김 전술 2개처럼 완전한 제로샷은 아니다.

[볼 것] A와 B의 역할이 다를 수 있다 (개수가 다름).

In [6]:
def label_text(l):
    return LABELS.at[l, 'label_text']

def kind(l):
    return LABELS.at[l, 'kind']

def role_table(d):
    rows = []
    for col in ['tactics', 'techniques']:
        cnt = d.explode(col).groupby([col, 'split']).size().unstack(fill_value=0).reindex(columns=['pool', 'val', 'test'], fill_value=0)
        for l, r in cnt.iterrows():
            if col == 'techniques' and l == 'BENIGN':
                continue
            if l == 'BENIGN':
                role, why = 'train', ''
            elif LABELS.at[l, 'role'] == 'held_out':                    # labels.csv의 기본 역할 (숨김 전술)
                role, why = 'held_out', 'held_tactic' if l in HELD_TACTICS else 'labels_csv'
            elif AUTO_HIDE and (r == 0).any():
                role, why = 'held_out', '|'.join(f'no_{sp}' for sp in ['pool', 'val', 'test'] if r[sp] == 0)
            else:
                role, why = 'train', ''
            rows.append(dict(label_id=l, kind=kind(l), name=LABELS.at[l, 'name'],
                             role=role, role_reason=why, pool=r.pool, val=r.val, test=r.test, label_text=label_text(l)))
    return pd.DataFrame(rows)

roles = {k: role_table(parts[k]) for k in 'AB'}
for k in 'AB':
    t = roles[k]
    print(f'===== {k}: 학습 {int((t.role == "train").sum())} / 숨김 {int((t.role == "held_out").sum())}')
    print(t.drop(columns='label_text').sort_values(['kind', 'role', 'pool'], ascending=[True, False, False]).to_string(index=False)); print()

===== A: 학습 25 / 숨김 24
 label_id      kind                                name     role     role_reason  pool  val  test
   BENIGN    benign                              Benign    train                  2842 1410  1441
   TA0007    tactic                           Discovery    train                  1348  669   794
   TA0003    tactic                         Persistence    train                    81   33    89
   TA0004    tactic                Privilege Escalation    train                    80   33    89
   TA0006    tactic                   Credential Access    train                    23    8     8
   TA0001    tactic                      Initial Access    train                    17    1    53
   TA0005    tactic                             Stealth    train                    17    1    53
   TA0002    tactic                           Execution    train                    13    1     1
   TA0009    tactic                          Collection    train                    13    5    

## 6. 검사

- 숨김 전술 2개가 pool·val에 없는지
- 문장(text) 중복 없는지
- 같은 세션에서 다른 칸 윈도우끼리 이벤트가 겹치지 않는지

In [7]:
for k in 'AB':
    d = parts[k]
    for sp in ['pool', 'val']:
        assert not d[d.split == sp].tactics.map(lambda ls: any(l in HELD_TACTICS for l in ls)).any(), f'{k} {sp}에 숨김 전술'
    assert d.text.is_unique
    for s, g in d.groupby('session'):
        g = g.sort_values('window'); sp = g.split.values; ww = g.window.values
        for i in range(len(g)):
            assert not ((np.abs(ww - ww[i]) < 5) & (sp != sp[i])).any(), f'겹침 남음: {s}'
print('검사 통과')

검사 통과


## 7. 방식·k별 파일 만들기 (전술 / 기법 / 전술+기법 × k)

**k마다 역할을 다시 정한다**: 5번에서 학습이었던 라벨도 pool에 k개가 안 되면 이 k에서는 숨김 (이유 `pool<k`). 그래서 k가 커질수록 학습 라벨이 줄고 숨김 라벨이 는다.

| 방식 | 정답 (`true_labels`) | train·val 후보 | test 후보 |
|---|---|---|---|
| `tactic` | 전술 (+ BENIGN) | 학습 전술 | 학습 + 숨김 전술 |
| `technique` | 기법 (+ BENIGN) | 학습 기법 | 학습 + 숨김 기법 |
| `both` | 전술 + 기법 (+ BENIGN) | 학습 전술 + 학습 기법 | 전부 |

정답은 후보 라벨에 있는 것만 넣는다 (숨김 라벨은 train·val 정답에서 빠짐).

**k-shot**: pool에서 seed로 섞고, **pool에 적은 라벨부터** 그 라벨이 붙은 윈도우를 k개 찰 때까지 담는다. 학습 라벨은 모두 pool에 k개 이상이라 항상 채워진다.

In [8]:
def mode_labels(row, mode):
    if mode == 'tactic':
        return row.tactics
    if mode == 'technique':
        return row.techniques
    return sorted(set(row.tactics) | set(row.techniques))

def sample_kshot(pool, train_ids, k, seed):
    order = pool.sample(frac=1, random_state=seed)
    has = {l: order['labels'].map(lambda ls: l in ls) for l in train_ids}
    chosen, cnt = [], Counter()
    for l in sorted(train_ids, key=lambda l: has[l].sum()):
        for ix in order.index[has[l]]:
            if cnt[l] >= k: break
            if ix not in chosen:
                chosen.append(ix); cnt.update([x for x in order.at[ix, 'labels'] if x in train_ids])
    return pool.loc[chosen], cnt

def to_gliclass(d, lids, text_of):
    return [{'text': r.text, 'all_labels': [text_of[l] for l in lids],
             'true_labels': [text_of[l] for l in r.labels if l in lids]} for r in d.itertuples()]

cols = ['text', 'labels', 'tactics', 'techniques', 'session', 'window', 'source', 'mixed', 'n_attack_events', 'n_benign_events', 'split']
for old in OUT.glob('*_*'):                       # 이전 결과 지우기
    if old.is_dir(): shutil.rmtree(old)

all_roles, summary = [], []
for v in 'AB':
    for mode in MODES:
        base_lab = roles[v] if mode == 'both' else roles[v][roles[v].kind.isin([mode, 'benign'])]
        d = parts[v].copy()
        d['labels'] = [[l for l in mode_labels(r, mode) if l in set(base_lab.label_id)] for r in d.itertuples()]
        for kk in KS:
            lab = base_lab.copy()
            short = (lab.role == 'train') & (lab.label_id != 'BENIGN') & (lab.pool < kk)
            lab.loc[short, 'role'] = 'held_out'                       # pool에 k개가 안 되면 이 k에서는 숨김
            lab.loc[short, 'role_reason'] = f'pool<{kk}'
            TRAIN = lab[lab.role == 'train'].label_id.tolist()
            HELD = lab[lab.role == 'held_out'].label_id.tolist()
            text_of = dict(zip(lab.label_id, lab.label_text))

            name = f'{v}_{mode}_k{kk}'
            base = OUT / name
            sp_dir = base / 'splits' / name; (sp_dir / 'kshot').mkdir(parents=True)
            lab.to_csv(base / 'labels.csv', index=False)
            all_roles.append(lab.assign(variant=v, mode=mode, k=kk))
            for sp in ['pool', 'val', 'test']:
                d[d.split == sp][cols].to_json(sp_dir / f'{sp}.jsonl', orient='records', lines=True, force_ascii=False)
            json.dump(to_gliclass(d[d.split == 'val'], TRAIN, text_of), open(sp_dir / 'val_gliclass.json', 'w'), ensure_ascii=False)
            json.dump(to_gliclass(d[d.split == 'test'], TRAIN + HELD, text_of), open(sp_dir / 'test_gliclass.json', 'w'), ensure_ascii=False)
            pool = d[d.split == 'pool']
            for sd in SEEDS:
                s, cnt = sample_kshot(pool, TRAIN, kk, sd)
                assert all(cnt[l] >= kk for l in TRAIN), (name, sd)
                json.dump(to_gliclass(s, TRAIN, text_of), open(sp_dir / 'kshot' / f'train_k{kk}_s{sd}.json', 'w'), ensure_ascii=False)
                summary.append(dict(variant=v, mode=mode, k=kk, seed=sd, windows=len(s)))
            print(f'{name:18s} 학습 {len(TRAIN):2d} / 숨김 {len(HELD):2d} (이 k에서 숨김으로 바뀐 것 {int(short.sum())})')

all_roles = pd.concat(all_roles, ignore_index=True)
all_roles.drop(columns='label_text').to_csv(OUT / 'roles_by_k.csv', index=False)
summary = pd.DataFrame(summary)

A_tactic_k1        학습 11 / 숨김  2 (이 k에서 숨김으로 바뀐 것 0)
A_tactic_k3        학습 10 / 숨김  3 (이 k에서 숨김으로 바뀐 것 1)


A_tactic_k5        학습 10 / 숨김  3 (이 k에서 숨김으로 바뀐 것 1)
A_tactic_k8        학습  9 / 숨김  4 (이 k에서 숨김으로 바뀐 것 2)


A_tactic_k16       학습  7 / 숨김  6 (이 k에서 숨김으로 바뀐 것 4)


A_technique_k1     학습 15 / 숨김 22 (이 k에서 숨김으로 바뀐 것 0)
A_technique_k3     학습 13 / 숨김 24 (이 k에서 숨김으로 바뀐 것 2)


A_technique_k5     학습 12 / 숨김 25 (이 k에서 숨김으로 바뀐 것 3)
A_technique_k8     학습  9 / 숨김 28 (이 k에서 숨김으로 바뀐 것 6)


A_technique_k16    학습  7 / 숨김 30 (이 k에서 숨김으로 바뀐 것 8)


... (8 outputs left out)

B_technique_k16    학습  8 / 숨김 29 (이 k에서 숨김으로 바뀐 것 7)


B_both_k1          학습 25 / 숨김 24 (이 k에서 숨김으로 바뀐 것 0)
B_both_k3          학습 18 / 숨김 31 (이 k에서 숨김으로 바뀐 것 7)


B_both_k5          학습 15 / 숨김 34 (이 k에서 숨김으로 바뀐 것 10)
B_both_k8          학습 14 / 숨김 35 (이 k에서 숨김으로 바뀐 것 11)


B_both_k16         학습 14 / 숨김 35 (이 k에서 숨김으로 바뀐 것 11)


## 8. k마다 학습 라벨 수, 숨김으로 바뀐 라벨

[볼 것] k가 커질수록 pool이 적은 라벨이 숨김으로 바뀐다.

In [9]:
n_train = all_roles[all_roles.role == 'train'].groupby(['variant', 'mode', 'k']).size().unstack('k')
print('학습 라벨 수 (Benign 포함)'); print(n_train.to_string())
moved = all_roles[all_roles.role_reason.str.startswith('pool<')]
first_k = moved[moved['mode'] == 'both'].groupby(['variant', 'label_id', 'name']).k.min().unstack('variant')
print('\n숨김으로 바뀌기 시작하는 k (both 기준):'); print(first_k.to_string())

학습 라벨 수 (Benign 포함)
k                  1   3   5   8   16
variant mode                         
A       both       25  22  21  17  13
        tactic     11  10  10   9   7
        technique  15  13  12   9   7
B       both       25  18  15  14  14
        tactic     11   9   8   7   7
        technique  15  10   8   8   8

숨김으로 바뀌기 시작하는 k (both 기준):
variant                                        A    B
label_id  name                                       
T1021     Remote Services                    3.0  3.0
T1021.008 Direct Cloud VM Connections        3.0  3.0
T1110     Brute Force                        5.0  3.0
T1530     Data from Cloud Storage           16.0  NaN
T1537     Transfer Data to Cloud Account     8.0  5.0
T1552     Unsecured Credentials             16.0  5.0
T1555     Credentials from Password Stores   8.0  3.0
T1555.006 Cloud Secrets Management Stores    8.0  3.0
TA0002    Execution                         16.0  3.0
TA0006    Credential Access                  NaN  8.0


[볼 것] 방식별 학습 파일 한 줄 예시

In [10]:
for mode in MODES:
    ex = json.load(open(OUT / f'B_{mode}_k5' / 'splits' / f'B_{mode}_k5' / 'kshot' / 'train_k5_s0.json'))[0]
    print(f'[B_{mode}_k5] 정답 {len(ex["true_labels"])}개 / 후보 {len(ex["all_labels"])}개:', [t.split(':')[0] for t in ex['true_labels']])
print('\nfinetuning.ipynb: DATA_DIR=data/v2/B_tactic_k16, VARIANT=B_tactic_k16, K_LIST=[16] 처럼 실행')

[B_tactic_k5] 정답 1개 / 후보 8개: ['Credential Access']
[B_technique_k5] 정답 4개 / 후보 8개: ['Valid Accounts', 'Cloud Accounts', 'Account Manipulation', 'Additional Cloud Credentials']
[B_both_k5] 정답 1개 / 후보 15개: ['Credential Access']

finetuning.ipynb: DATA_DIR=data/v2/B_tactic_k16, VARIANT=B_tactic_k16, K_LIST=[16] 처럼 실행
